In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://fleet-data@fleetdatalake123.dfs.core.windows.net/raw/vehicle_tracking/vehicletracking_merged.csv")

df.show(10)

+---------+---------+---------+---------+------------+-------------+-------------------+--------+---------+-----+--------------------+----------+
|VehicleId|VehicleNo| Latitude|Longitude|Latitude_Alt|Longitude_Alt|           DateTime|Ignition|GPSstatus|speed|        LocationName|drivername|
+---------+---------+---------+---------+------------+-------------+-------------------+--------+---------+-----+--------------------+----------+
|      123|TG28T3177|17.963202|80.857597|      17.963|       80.858|2026-04-01 09:47:42|     Off|       On|  0.0|Ramanujavaram, Ma...|   driver1|
|      123|TG28T3177|17.963202|80.857597|      17.963|       80.858|2026-04-01 09:52:42|     Off|       On|  0.0|Ramanujavaram, Ma...|   driver1|
|      123|TG28T3177|17.963202|80.857597|      17.963|       80.858|2026-04-01 09:57:42|     Off|       On|  0.0|Ramanujavaram, Ma...|   driver1|
|      123|TG28T3177|17.963202|80.857597|      17.963|       80.858|2026-04-01 10:02:42|     Off|       On|  0.0|Ramanujavar

In [0]:
spark.conf.set(
    "fs.azure.account.key.fleetdatalake123.dfs.core.windows.net",
    "zdkQVa67CDTnCiPNpR8Zsh6Adim1LGdmBt9mgLCPz8BmMpdim4kGj/xnS5Yh555tPux/ezF1sjN/+AStAhRNKA=="
)

In [0]:
df.count()

498526

In [0]:
df.printSchema()

root
 |-- VehicleId: integer (nullable = true)
 |-- VehicleNo: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Latitude_Alt: double (nullable = true)
 |-- Longitude_Alt: double (nullable = true)
 |-- DateTime: timestamp (nullable = true)
 |-- Ignition: string (nullable = true)
 |-- GPSstatus: string (nullable = true)
 |-- speed: double (nullable = true)
 |-- LocationName: string (nullable = true)
 |-- drivername: string (nullable = true)



In [0]:
# Import required functions from PySpark
from pyspark.sql.functions import current_timestamp, input_file_name

# Add metadata columns to raw dataset
# _ingest_ts → captures when data was ingested into system
# _source_file → captures source file path for traceability
bronze_df = df.withColumn("_ingest_ts", current_timestamp()) \
              .withColumn("_source_file", input_file_name())

# Write DataFrame to Delta table (Bronze Layer)
# format("delta") → enables Delta Lake features (ACID, versioning)
# mode("overwrite") → replaces table if it already exists
# saveAsTable → creates a managed table inside Databricks metastore
bronze_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_vehicle_tracking")

In [0]:
# Write the DataFrame to a Delta table (Bronze Layer)
# Use Delta Lake format → supports ACID transactions, versioning, and reliability
# Overwrite mode → replaces the table if it already exists (useful during development)
# saveAsTable → creates a managed table in Databricks metastore
# Table name: bronze_vehicle_tracking (represents raw ingested data layer)

bronze_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_vehicle_tracking")

In [0]:
# Read the Bronze table to verify data was written successfully
spark.table("bronze_vehicle_tracking").show(5)

+---------+---------+---------+---------+------------+-------------+-------------------+--------+---------+-----+--------------------+----------+--------------------+--------------------+
|VehicleId|VehicleNo| Latitude|Longitude|Latitude_Alt|Longitude_Alt|           DateTime|Ignition|GPSstatus|speed|        LocationName|drivername|          _ingest_ts|        _source_file|
+---------+---------+---------+---------+------------+-------------+-------------------+--------+---------+-----+--------------------+----------+--------------------+--------------------+
|      123|TG28T3177|17.963202|80.857597|      17.963|       80.858|2026-04-01 09:47:42|     Off|       On|  0.0|Ramanujavaram, Ma...|   driver1|2026-04-27 19:24:...|abfss://fleet-dat...|
|      123|TG28T3177|17.963202|80.857597|      17.963|       80.858|2026-04-01 09:52:42|     Off|       On|  0.0|Ramanujavaram, Ma...|   driver1|2026-04-27 19:24:...|abfss://fleet-dat...|
|      123|TG28T3177|17.963202|80.857597|      17.963|      